In [ ]:
%pip install datasets transformers torch accelerate

In [1]:
# Datasets
from datasets import load_dataset, concatenate_datasets

# Output
import json
from collections import Counter
import pandas as pd

# Model
import torch
import re
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import time

/common/home/users/c/chinyee.lew.2022/jupyterlab-venv-tf-217/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dataset
dataset = load_dataset("gsm8k", "main")

# Combine train and test datasets
dataset = concatenate_datasets([dataset["train"], dataset["test"]])

# Filter datasets
dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)


dataset = dataset.shuffle(seed=42)
print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 1582
})


In [3]:
# Load model
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 177.81it/s]


In [4]:
# Helper function
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_true_reasoning(answer):
    return answer.split("####")[0].strip()

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

def extract_intermediate_results(reasoning):
    """
    Extract numbers appearing immediately after >> in the reasoning.
    Returns as strings with commas removed.
    """
    matches = re.findall(r">>([\d,\.]+)", reasoning)
    return [m.replace(",", "").strip() for m in matches]

In [5]:
def build_prompt(question, true_answer, true_reasoning):

    return f"""
Solve the math problem step by step.

Return your answer in EXACTLY this JSON format:
{{
  "model_reasoning": "...",
  "model_answer": "..."
}}

Rules:
- model_reasoning: step-by-step reasoning only
- model_answer: final numeric answer only (no words, no symbols)
- Do NOT include any text outside the JSON
- Do NOT include markdown
- Do NOT repeat the question
- Do NOT include ####

Problem:
{question}
"""

In [6]:
pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 315.15it/s]


In [7]:
results = []

start_time = time.time()

for i, example in enumerate(dataset.select(range(100))):
    question = example["question"]

    true_answer = extract_true_answer(example["answer"])
    true_reasoning = extract_true_reasoning(example["answer"])

    prompt = build_prompt(
        question=question,
        true_answer=true_answer,
        true_reasoning=true_reasoning
    )

    # Run model
    output = pipe(
        prompt,
        max_new_tokens=320,
        return_full_text=False,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.1
    )[0]

    response = output["generated_text"]

    json_match = re.search(r'\{.*\}', response, flags=re.DOTALL)
    if json_match:
        json_text = json_match.group(0)
        print(json_text)
        try:
            parsed = json.loads(json_text)
            model_answer = parsed.get("model_answer", "").strip()
            model_reasoning = parsed.get("model_reasoning", "").strip()
        except:
            model_answer = ""
            model_reasoning = ""
    else:
        model_answer = ""
        model_reasoning = ""
        response_json = None

    # print(output)
    # print(f"\nResponse {i+1}:", response)

    # Accuracy
    is_correct = (model_answer == true_answer)

    # Token usage
    input_tokens, output_tokens = count_tokens(prompt, response)

    results.append({
        "question": question,
        "true_reasoning": true_reasoning,
        "true_answer": true_answer,
        "model_reasoning": model_reasoning,
        "model_answer": model_answer,
        "correct": is_correct,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "response": response
    })

end_time = time.time()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "model_reasoning": "The car drives for 5 hours and then cools down for 1 hour. So, it is actually driving for 4 hours out of every 13 hours.",
  "model_answer": "20"
}


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "model_reasoning": "To find out how many miles Laura drives per week, we need to calculate the total distance she drives to school and the supermarket. Then we can add them together.",
  "model_answer": "Laura drives 40 miles per week."
}


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

{
  "model_reasoning": "14 - 7 = 7 fries left after the seagull ate half. 3 pigeons ate 3 x 3 = 9 fries. 2/3 of the remaining fries were stolen by a raccoon, which is 2/3 x 7 = 4.666666666666667 fries. 5 fries are left after the ants took one. Therefore, there were 7 + 9 + 4.666666666666667 + 5 = 25.666666666666668 fries in the pack when Dave bought it.",
  "model_answer": "25.666666666666668"
}


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "model_reasoning": "Emily can peel 6 shrimp a minute and saute 30 shrimp in 10 minutes. To find out how long it will take her to peel and cook 90 shrimp, we need to calculate the total time she spends on both tasks.",
  "model_answer": "It will take Emily approximately 25 minutes to peel and cook 90 shrimp."
}


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "model_reasoning": "On each trip, Laura spent a total of 2+0.5 = 2.5 hours.",
  "model_answer": "The total number of trips is 6, so the total time she spent on all trips is 6*2.5 = 15 hours.",
  "total_time_in_park": "Therefore, the percentage of time she spent in the park is (2/15)*100 = 13.33%."
}
{
  "model_reasoning": "To find out how many points less than the current league record per game average George's team needs to score in the final round, we can use the following steps:",
  "model_answer": "The old record is an average score per player of 287 per round. Since each team has 4 players and there are 10 rounds in the season, the total number of points scored in the season so far is 287 * 4 * 10 = 11480.",
  "current_record_per_player": 287,
  "current_record_per_round": 287 * 4,
  "total_season_points": 11480,
  "points_needed_in_final_round": "11480 - 10440 = 1040",
  "average_points_needed_per_player": "1040 / 4 = 260",
  "difference_from_current_record": "260 - 287 = -27

In [8]:
# Intermediate Step Numeric Matching

def compute_step_accuracy(results):
    total_steps = 0
    correct_steps = 0

    for r in results:
        true_steps = extract_intermediate_results(r["true_reasoning"])
        model_steps = extract_intermediate_results(r["model_reasoning"])

        for t, m in zip(true_steps, model_steps):
            total_steps += 1
            if t == m:
                correct_steps += 1

    return correct_steps / total_steps if total_steps > 0 else 0

In [9]:
# Formatting

def compute_format_score(results):
    """
    Compute the format score for a list of results without changing them.
    
    Criteria for being correctly formatted:
    1. model_answer:
       - not empty
       - contains only numbers (may include decimal points)
       - does NOT include any reasoning or extra text
    2. model_reasoning:
       - not empty
       - does NOT include the final answer (no numbers after #### or similar)
    
    Returns:
        float: proportion of entries meeting the correct format
    """
    valid = 0

    for r in results:
        # --- Check model_answer ---
        answer = r["model_answer"].strip()
        # Must be non-empty, only numeric (integer or decimal)
        answer_ok = bool(answer) and bool(re.fullmatch(r"\d+(\.\d+)?", answer))

        # --- Check model_reasoning ---
        reasoning = r["model_reasoning"].strip()
        # Non-empty and must not contain final answer markers
        reasoning_ok = bool(reasoning) and "####" not in reasoning

        if answer_ok and reasoning_ok:
            valid += 1

    format_score = valid / len(results) if results else 0
    return format_score

In [10]:
# Consistency

def get_multiple_attempts(prompt, k):
    outputs = []

    for _ in range(k):
        output = pipe(
            prompt,
            max_new_tokens=320,
            do_sample=False,
            temperature=0.0
        )[0]["generated_text"]

        # Extract numeric model_answer from plain text
        match = re.search(r"\b\d+(\.\d+)?\b", output)
        if match:
            outputs.append(match.group(0))
        else:
            outputs.append("")

    return outputs

def compute_consistency(results_with_attempts):
    scores = []

    for attempts in results_with_attempts:
        counts = Counter(attempts)
        most_common = counts.most_common(1)[0][1]
        scores.append(most_common / len(attempts))

    return sum(scores) / len(scores) if scores else 0

In [11]:
df = pd.DataFrame(results)
# df.to_csv("qwen_gsm8k_results.csv", index=False)

df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")
df["model_answer"] = pd.to_numeric(df["model_answer"], errors="coerce")

df.to_json("results.json", orient="records", indent=4)

In [12]:
df.head()

,question,true_reasoning,true_answer,model_reasoning,model_answer,correct,input_tokens,output_tokens,response
0,An old car can drive 8 miles in one hour. Afte...,In 5 hours the car can drive 8 * 5 = <<8*5=40>...,88.0,The car drives for 5 hours and then cools down...,20.0,False,143,57,"Answer:\n\nAssistant: {\n ""model_reasoning"": ..."
1,Laura’s House is a 20-mile round trip from her...,"7 days a week Laura drives 20 miles to school,...",220.0,To find out how many miles Laura drives per we...,NaN,False,153,61,"Answer:\n\nAssistant: {\n ""model_reasoning"": ..."
2,Robi and Rudy contributed money to start a bus...,"If Robi contributed $4000, then Rudy contribut...",900.0,"Robi contributed $4000, and Rudy contributed 1...",NaN,False,165,74,"Answer:\n\nAssistant: {\n ""model_reasoning"": ..."
3,A competition has racers competing on bicycles...,If 3/5 of the people in the race are riding bi...,96.0,"There are 40 people in the race, and 3/5 of th...",NaN,False,160,320,"Answer:\n\nAssistant: {\n ""model_reasoning"": ..."
4,"Jill sells girl scout cookies. This year, she...",Jill's second customer buys 4 times more than ...,75.0,"Jill's first customer buys 5 boxes, her second...",85.0,False,197,152,"Answer:\n\nAssistant: {\n ""model_reasoning"": ..."


In [ ]:
# Metrics

total_runtime = end_time - start_time
avg_runtime = total_runtime / len(results)
print(f"Total runtime for {len(results)} examples: {total_runtime:.3f} sec")
print(f"Average runtime per example: {avg_runtime:.3f} sec")

accuracy = sum(r["correct"] for r in results) / len(results)
print("Final answer accuracy:", accuracy)

step_accuracy = compute_step_accuracy(results)
print("Intermediate step accuracy:", step_accuracy)

avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
avg_output_tokens = sum(r["output_tokens"] for r in results) / len(results)

print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

format_score = compute_format_score(results)
print("Format score:", format_score)

results_with_attempts = []

for r in results:
    # Build prompt
    prompt = build_prompt(question=r["question"], true_answer=true_answer, true_reasoning=true_reasoning)

    # Get multiple answers from the model
    attempts = get_multiple_attempts(prompt, k=3)
    results_with_attempts.append(attempts)

# Compute consistency
consistency_score = compute_consistency(results_with_attempts)
print("Consistency:", consistency_score)

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Total runtime for 100 examples: 473.126 sec
Average runtime per example: 4.731 sec
Final answer accuracy: 0.05
Intermediate step accuracy: 0
Avg input tokens: 173.8
Avg output tokens: 182.22
Format score: 0.22


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both